# FinAI – EDA & Lightweight Modeling Notebook

This notebook is designed to be **runnable end-to-end** in a local dev environment using the
FinAI backend, while also providing a **CSV-based fallback** if the API is not reachable.

## How to use

1. **Start the backend API** (recommended)
   - From the `apps/api` folder:
     ```bash
     pip install -r requirements.txt
     uvicorn app.main:app --reload
     ```
   - By default this serves at `http://localhost:8000`.

2. **Choose your data source** (in the next cell)
   - `MODE = "backend"` (default): fetch OHLC history and AI insights from the API.
   - `MODE = "csv"`: download or reuse a CSV into `notebooks/data/` and load from disk.

3. **Install notebook dependencies**
   - The backend already pins:
     - `pandas`, `numpy`, `matplotlib`, `scikit-learn`
   - If you're running this notebook standalone, you can install them with:
     ```bash
     pip install pandas numpy matplotlib scikit-learn
     ```

4. **Run cells top to bottom**
   - The notebook will:
     - Load OHLC data for a chosen symbol (default: `AAPL`).
     - Compute basic indicators (MA, Bollinger, RSI).
     - Fit a simple linear regression on lag features.
     - Compute permutation importance.
     - Plot forecast vs actuals and show a simple anomaly view.

> **Important:** All analytics and models here are for **educational purposes only**.
> Nothing in this notebook is financial advice or a recommendation to buy or sell any asset.

In [ ]:
MODE = "backend"  # "backend" or "csv"
API_BASE = "http://localhost:8000"  # Adjust if your backend runs elsewhere
SYMBOL = "AAPL"
ASSET_TYPE = "stock"  # "stock" or "crypto"

In [ ]:
import json
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

plt.style.use("seaborn-v0_8-darkgrid")

DATA_DIR = Path("notebooks/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Mode: {MODE}")
print(f"API base: {API_BASE}")

## Load OHLC data

We try the backend first (if `MODE == "backend"`). If that fails, we fall back to a CSV file.

For the CSV fallback we use Yahoo-style columns: `Date, Open, High, Low, Close, Volume`.
You can either:

- Let the notebook auto-download a sample AAPL CSV from a public GitHub dataset, or
- Replace the file in `notebooks/data/custom_prices.csv` with your own history.

In [ ]:
def load_from_backend(symbol: str, asset_type: str) -> pd.DataFrame:
    """Load daily bars via FinAI backend history + enrich with AI insights if available."""
    history_url = f"{API_BASE}/api/v1/history"
    params = {
        "symbol": symbol,
        "asset_type": asset_type,
        "interval": "1d",
        "range": "3mo",
    }
    resp = requests.get(history_url, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    bars = pd.DataFrame(data["bars"])
    bars["ts"] = pd.to_datetime(bars["ts"], utc=True)
    bars = bars.sort_values("ts").reset_index(drop=True)
    bars = bars.rename(columns={
        "ts": "Date",
        "open": "Open",
        "high": "High",
        "low": "Low",
        "close": "Close",
        "volume": "Volume",
    })
    return bars


def load_from_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"], utc=True)
    df = df.sort_values("Date").reset_index(drop=True)
    return df


def ensure_sample_csv(path: Path) -> None:
    if path.exists():
        return
    url = "https://raw.githubusercontent.com/plotly/datasets/master/finance-charts-apple.csv"
    print(f"Downloading sample CSV to {path} ...")
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    raw = pd.read_csv(StringIO(resp.text))
    # The Plotly dataset uses column names like AAPL.Open; normalize them.
    if "AAPL.Open" in raw.columns:
        df = pd.DataFrame({
            "Date": pd.to_datetime(raw["Date"], utc=True),
            "Open": raw["AAPL.Open"],
            "High": raw["AAPL.High"],
            "Low": raw["AAPL.Low"],
            "Close": raw["AAPL.Close"],
            "Volume": raw.get("AAPL.Volume", 0),
        })
    else:
        df = raw
    df.to_csv(path, index=False)


def get_prices(symbol: str, asset_type: str) -> pd.DataFrame:
    if MODE == "backend":
        try:
            df = load_from_backend(symbol, asset_type)
            print(f"Loaded {len(df)} rows from backend for {symbol}.")
            return df
        except Exception as exc:
            print(f"Backend load failed ({exc}); falling back to CSV mode.")

    # CSV fallback
    csv_path = DATA_DIR / "sample_prices.csv"
    try:
        ensure_sample_csv(csv_path)
    except Exception as exc:
        raise RuntimeError(
            "Could not download or read sample CSV. Please place a CSV at "
            f"{csv_path} with columns Date, Open, High, Low, Close, Volume."
        ) from exc

    df = load_from_csv(csv_path)
    print(f"Loaded {len(df)} rows from CSV fallback at {csv_path}.")
    return df


prices = get_prices(SYMBOL, ASSET_TYPE)
prices.tail()

## Basic EDA

We will:

- Plot closing prices.
- Compute daily returns.
- Compute a simple moving average (MA20) and Bollinger Bands.
- Derive a simple RSI metric for the last point.

In [ ]:
df = prices.copy()
df = df.set_index("Date").sort_index()
df["Return"] = df["Close"].pct_change()

window = 20
df["MA20"] = df["Close"].rolling(window).mean()
df["STD20"] = df["Close"].rolling(window).std(ddof=0)
df["UpperBB"] = df["MA20"] + 2 * df["STD20"]
df["LowerBB"] = df["MA20"] - 2 * df["STD20"]

def simple_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df["RSI14"] = simple_rsi(df["Close"], period=14)

fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

axes[0].plot(df.index, df["Close"], label="Close", color="white")
axes[0].plot(df.index, df["MA20"], label="MA20", color="cyan")
axes[0].fill_between(df.index, df["UpperBB"], df["LowerBB"], color="cyan", alpha=0.1)
axes[0].set_ylabel("Price")
axes[0].legend(loc="upper left")

axes[1].plot(df.index, df["Return"] * 100, label="Daily return %", color="orange")
axes[1].axhline(0, color="grey", linewidth=0.7)
axes[1].set_ylabel("Return %")

axes[2].plot(df.index, df["RSI14"], label="RSI14", color="magenta")
axes[2].axhline(70, color="red", linestyle="--", linewidth=0.7)
axes[2].axhline(30, color="green", linestyle="--", linewidth=0.7)
axes[2].set_ylabel("RSI")
axes[2].set_xlabel("Date")
axes[2].legend(loc="upper left")

fig.suptitle(f"{SYMBOL} – Price, Returns, and RSI", color="white")
plt.tight_layout()
plt.show()

print("Latest RSI:", float(df["RSI14"].iloc[-1]))

## Lightweight forecasting model (Linear Regression on lag features)

We build a small design matrix where each row uses the previous `N` closes to predict the
next close:

\begin{align}
y_t &= \text{Close}_t \\
X_t &= [\text{Close}_{t-1}, \dots, \text{Close}_{t-N}]
\end{align}

Then we:

- Fit a `LinearRegression` model.
- Compute in-sample RMSE.
- Roll the model forward for a short forecast horizon (e.g., 7 steps).
- Plot forecast vs actual for the tail of the series.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

close = df["Close"].dropna().values.astype(float)
n_lags = 5
horizon = 7

if len(close) <= n_lags + horizon:
    raise RuntimeError("Not enough data to train the toy model. Try a longer history range.")

X = []
y = []
for i in range(n_lags, len(close)):
    X.append(close[i - n_lags : i])
    y.append(close[i])

X = np.asarray(X)
y = np.asarray(y)

model = LinearRegression()
model.fit(X, y)
pred_in_sample = model.predict(X)
rmse = np.sqrt(mean_squared_error(y, pred_in_sample))
print(f"In-sample RMSE: {rmse:.4f}")

# Build a simple iterative forecast for the next horizon steps
last_lags = list(close[-n_lags:])
forecast_values = []
for _ in range(horizon):
    x_next = np.asarray(last_lags).reshape(1, -1)
    y_next = float(model.predict(x_next)[0])
    forecast_values.append(y_next)
    last_lags.pop(0)
    last_lags.append(y_next)

fc_index = pd.date_range(df.index[-1] + pd.Timedelta(days=1), periods=horizon, freq="D")
fc_series = pd.Series(forecast_values, index=fc_index, name="Forecast")

fig, ax = plt.subplots(figsize=(10, 4))
tail_actual = df["Close"].iloc[-60:]
tail_actual.plot(ax=ax, label="Actual", color="white")
fc_series.plot(ax=ax, label="Forecast", color="cyan", linestyle="--")
ax.set_title(f"{SYMBOL} – Toy linear forecast (lag {n_lags})")
ax.legend()
plt.show()

## Permutation importance (feature view)

To get a feel for which lags matter most, we compute a simple **permutation importance**:

1. Compute baseline MSE on the training set.
2. For each column (lag feature), randomly permute its values and recompute MSE.
3. Importance \( \approx \text{MSE}_{\text{perm}} - \text{MSE}_{\text{baseline}} \).

This is purely illustrative but gives a sense of whether closer lags dominate the model.

In [ ]:
rng = np.random.default_rng(42)
baseline_mse = mean_squared_error(y, pred_in_sample)
importances = []

for j in range(X.shape[1]):
    X_perm = X.copy()
    X_perm[:, j] = rng.permutation(X_perm[:, j])
    y_perm_pred = model.predict(X_perm)
    mse_perm = mean_squared_error(y, y_perm_pred)
    importances.append(mse_perm - baseline_mse)

lags = [f"lag_{i+1}" for i in range(n_lags)][::-1]  # lag_1 ~ most recent
imp_series = pd.Series(importances[::-1], index=lags)

imp_series.sort_values().plot(kind="barh", figsize=(6, 4), color="#38bdf8")
plt.title("Permutation importance (higher = more important)")
plt.xlabel("Δ MSE")
plt.tight_layout()
plt.show()

imp_series

## Simple anomaly view on returns

We mark returns that are more than 2 standard deviations away from the mean as "anomalies".
This is intentionally simple – the production backend uses an Isolation Forest-based approach,
but the goal here is to make the idea easy to see in a single chart.

In [ ]:
returns = df["Return"].dropna()
mu = returns.mean()
sigma = returns.std(ddof=0)
threshold = 2 * sigma
anomalies = (returns - mu).abs() > threshold

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(returns.index, returns * 100, color="white", label="Daily return %")
ax.scatter(
    returns.index[anomalies],
    (returns[anomalies] * 100),
    color="red",
    label="Anomaly (>2σ)",
)
ax.axhline(mu * 100, color="grey", linestyle="--", linewidth=0.7, label="Mean")
ax.set_title(f"{SYMBOL} – Simple return anomalies")
ax.set_ylabel("Return (%)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean return: {mu:.4f}, sigma: {sigma:.4f}")
print(f"Anomaly days: {anomalies.sum()} / {len(returns)}")

## Wrap-up

What we achieved in this notebook:

- Cleaned and explored OHLC data for a single symbol.
- Computed MA/Bollinger/RSI indicators to complement the dashboard.
- Trained a very simple linear model on lag features.
- Derived permutation importance to see which lags matter most.
- Built a toy anomaly detector directly on returns.

All of this is intentionally simple and **CPU-friendly**, mirroring the design choices in the
backend AI endpoints. For production-grade modeling you would typically:

- Use more robust cross-validation and out-of-sample evaluation.
- Incorporate multiple symbols and macro features.
- Use probabilistic or more expressive models (e.g., gradient boosting).

But even this lightweight setup is enough to power the **AI Insights** panel in the FinAI UI.

> Again: This is **not** investment advice. Treat the outputs as educational analytics only.